## ✅ Task 5 – Predictive Model 🚀
**Goal:** Predict `TotalPrice` using a simple Linear Regression model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder

# ── Load dataset ──────────────────────────────────────────────────────────
df = pd.read_excel('data.xlsx')

# Data cleaning (from Task 2)
df['CouponCode'] = df['CouponCode'].fillna('No Coupon')
df.drop_duplicates(inplace=True)
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.month_name()
df['DayOfWeek'] = df['Date'].dt.day_name()
str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].apply(lambda c: c.str.strip())

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────
model_df = df.copy()

le = LabelEncoder()
model_df['Product_enc']       = le.fit_transform(model_df['Product'])
model_df['PaymentMethod_enc'] = le.fit_transform(model_df['PaymentMethod'])
model_df['HasCoupon']         = (model_df['CouponCode'] != 'No Coupon').astype(int)

features = ['Quantity', 'UnitPrice', 'ItemsInCart',
            'Product_enc', 'PaymentMethod_enc', 'HasCoupon']
target   = 'TotalPrice'

X = model_df[features]
y = model_df[target]

# ── Train / Test split ────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ── Train model ───────────────────────────────────────────────────────────
model = LinearRegression()
model.fit(X_train, y_train)

# ── Evaluate ──────────────────────────────────────────────────────────────
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print(f"Mean Absolute Error : ${mae:.2f}")
print(f"R² Score            : {r2:.4f}")
print()
print("Feature Coefficients:")
for feat, coef in zip(features, model.coef_):
    print(f"  {feat:<22}: {coef:.4f}")

In [ ]:
# ── Actual vs Predicted plot ──────────────────────────────────────────────
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred, alpha=0.4, color='steelblue', label='Predictions')
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'r--', label='Perfect fit')
plt.xlabel('Actual Total Price ($)')
plt.ylabel('Predicted Total Price ($)')
plt.title('Linear Regression – Actual vs Predicted')
plt.legend()
plt.tight_layout()
plt.show()

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print(f"\nModel Summary: R² = {r2:.2%} of variance explained.")
print(f"Mean Squared Error (MSE) : ${mse:.2f} (squared)")
print(f"Root Mean Squared Error (RMSE): ${rmse:.2f} (average prediction error)")